In [29]:
# WARNING будет загружен датасет и сами модели
!dvc pull

Everything is up to date.


# Результаты

## EDA

Максимальный размер изображения 500x500 пискелей. Наименьший 112x500 (почему так? за что?).

Ближайшая степень двойки это 512, но если мы будем доводить 112 до 512, это превратится в маленькую полосочку и огромную пустоту.

Однако, для заполнения этой пустоты будем использовать скейлинг. Мы добьем недостающий размер симметричным изображением (даже если придется дублировать несколько раз (но это будет явно сложнее)).

- данные изменени

## Шаг 1

Решаем задачу Language Modelling

Embedding vs tokenizer

## Шаг 2

Решаем задачу Image 2 Caption

Вроде решили, но...

...аномалия.

Описать возможные причины и пути решения

## Вывод

Казалось проще чем в реализации


In [2]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import seaborn as sns
import statsforecast.models as nixtla
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler
from sklearn.utils.validation import check_is_fitted
from statsforecast import StatsForecast
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from sklearn.exceptions import NotFittedError
from model_utils import get_images, save_and_pad_image
from transformers import RobertaTokenizer

warnings.filterwarnings("ignore")

In [ ]:
def print_min_max_sizes(images):
    sizes = dict()
    for image in images:
        if image.shape in sizes:
            sizes[image.shape] += 1
        else:
            sizes[image.shape] = 1

    sorted_sizes = sorted(list(sizes.keys()), key=lambda x: (x[0], x[1]))
    print("Наименьший размер изображения изображение:", sorted_sizes[0])
    print("Наибольший размер изображения изображение:", sorted_sizes[-1])


images = get_images()
print_min_max_sizes(images.values())

target_folder = "./dataset/flickr30k_images_resized/"
for name, image in images.items():
    save_and_pad_image(target_folder, name, image)

print_min_max_sizes(get_images(folder_path=target_folder).values())


In [12]:
captions = pd.read_csv("./dataset/captions.txt")
print(captions.info())
captions.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158915 entries, 0 to 158914
Data columns (total 3 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   image_name      158915 non-null  object
 1   comment_number  158915 non-null  int64 
 2   comment         158915 non-null  object
dtypes: int64(1), object(2)
memory usage: 3.6+ MB
None


,image_name,comment_number,comment
0,1000092795.jpg,0,Two young guys with shaggy hair look at their ...
1,1000092795.jpg,1,Two young White males are outside near many b...
2,1000092795.jpg,2,Two men in green shirts are standing in a yard .
3,1000092795.jpg,3,A man in a blue shirt standing in a garden .
4,1000092795.jpg,4,Two friends enjoy time spent together .


In [18]:
max_length = captions["comment"].map(len).max()


def print_length_info(data, suffix):
    print(f"Длины сообщений ({suffix}): max={data["comment"].map(len).max()} mean={data["comment"].map(len).mean()}")


print_length_info(captions, "все комментарии")
for i in range(5):
    print_length_info(captions[captions["comment_number"] == i], f"номер комментария {i}")

Длины сообщений (все комментарии): max=403 mean=64.04440109492496
Длины сообщений (номер комментария 0): max=403 mean=94.41824245665921
Длины сообщений (номер комментария 1): max=215 mean=72.832898090174
Длины сообщений (номер комментария 2): max=175 mean=60.779441839977345
Длины сообщений (номер комментария 3): max=168 mean=51.07541767611616
Длины сообщений (номер комментария 4): max=179 mean=41.116005411698076


np.int64(403)

In [15]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
tokenizer(captions["comment"].iloc[0], padding='max_length', max_length=max_length, return_tensors="pt")

{'input_ids': tensor([[   0, 9058,  664, 1669,   19, 1481, 1073, 4740, 2549,  356,   23,   49,
         1420,  150, 7209,   66,   11,    5, 6993,  479,    2,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,